# Ingest Superstore Hyper Extracts
Loads the Tableau Hyper extracts embedded in `Super Sample Superstore.twbx` into Unity Catalog Delta tables:
* `serverless_stable_dvmvgw_catalog.genie.superstore_orders` (Orders already joined to Returns by Tableau; expected 12,420 rows)
* `serverless_stable_dvmvgw_catalog.genie.superstore_toc_annotations` (expected 4 rows)

Column names are snake_cased. Writes use mode `error`, so existing tables are never overwritten.

In [0]:
%pip install -q tableauhyperapi

In [0]:
import zipfile, re, datetime
from tableauhyperapi import HyperProcess, Telemetry, Connection
from pyspark.sql.types import StructType, StructField, StringType, DateType, DoubleType, LongType

src = "/Volumes/serverless_stable_dvmvgw_catalog/genie/marsh/tableau/Super Sample Superstore.twbx"
out = "/tmp/superstore_twbx"
with zipfile.ZipFile(src) as z:
    z.extractall(out)
base = f"{out}/Data/Extracts/"

targets = {
    "Sample _ Superstore.hyper": "serverless_stable_dvmvgw_catalog.genie.superstore_orders",
    "federated_1yv3g051gjjnme1dofc3r1.hyper": "serverless_stable_dvmvgw_catalog.genie.superstore_toc_annotations",
}
tmap = {"TEXT": StringType(), "DATE": DateType(), "DOUBLE": DoubleType(), "BIG_INT": LongType()}
snake = lambda n: re.sub(r"[^a-z0-9]+", "_", n.lower()).strip("_")

# Hyper files have no Spark reader, so rows are read on the driver (small extracts) and converted to a Spark DataFrame.
with HyperProcess(telemetry=Telemetry.DO_NOT_SEND_USAGE_DATA_TO_TABLEAU) as hp:
    for f, tgt in targets.items():
        with Connection(endpoint=hp.endpoint, database=base + f) as c:
            t = c.catalog.get_table_names("Extract")[0]
            d = c.catalog.get_table_definition(t)
            schema = StructType([StructField(snake(col.name.unescaped), tmap[str(col.type)], True) for col in d.columns])
            rows = [tuple(datetime.date(v.year, v.month, v.day) if v is not None and hasattr(v, "to_date") else v for v in r)
                    for r in c.execute_list_query(f"SELECT * FROM {t}")]
        spark.createDataFrame(rows, schema).write.mode("error").saveAsTable(tgt)
        spark.sql(f"COMMENT ON TABLE {tgt} IS 'Loaded from Tableau extract {f} (Super Sample Superstore.twbx)'")
        print(tgt, spark.table(tgt).count())

In [0]:
%sql
SELECT 'superstore_orders' AS table_name, COUNT(*) AS row_count FROM serverless_stable_dvmvgw_catalog.genie.superstore_orders
UNION ALL
SELECT 'superstore_toc_annotations', COUNT(*) FROM serverless_stable_dvmvgw_catalog.genie.superstore_toc_annotations